# 심화 미션: 스마트팜 온실 출하 기록
- 상황: 선별대에서 도장을 찍기 전에 등외를 미리 알고 싶다
- 목표: 오늘 배운 순서를 다른 데이터로 혼자 한 바퀴 돌린다

### 용어 풀이 - 온실에서 쓰는 말

| 말 | 뜻 |
|---|---|
| 상품 / 등외 | 선별대에서 붙이는 판정. 등외는 제값에 못 파는 것 |
| 양액 (EC) | 물에 녹인 거름. 그 진하기를 dS/m 라는 단위로 잰다 |
| 산도 (pH) | 산성인지 알칼리성인지. 7이 중간이고 낮을수록 산성 |
| 토양 수분 | 흙에 물기가 얼마나 있는지 (%) |
| 야간 최저 기온 | 밤에 가장 낮았던 기온. 작물이 스트레스를 받는 지점 |

## 공통 준비

In [1]:
import pandas as pd

df = pd.read_csv("../../data/day03_greenhouse.csv")

## Q1. 파일 열고 크기 확인하기

> 파일을 여니 표가 뜬다. 서 반장이 묻는다 — "넉 달치인데, 상자가 몇 개나 되죠?"

**문제** — 파일을 표로 불러오고, 몇 건이 몇 열로 되어 있는지 확인한다.

In [2]:
print(df.shape)

(2000, 13)


2,000건 × 13열. 넉 달 동안 상자 2,000개가 나갔다는 뜻이다.<br>
오늘 다룬 공정 데이터는 1,567행 × 592열이었다. **줄 수는 비슷한데 열이 훨씬 적다.**<br>
그래서 "무엇을 넣을지 고르는 일"이 이 데이터에서는 거의 없다.

## Q2. 등외는 얼마나 드문가

> "등외가 요즘 늘었다"는 말을 들었는데, 정작 몇 건인지는 아무도 세어본 적이 없다.

**문제** — 상품·등외 건수와 등외 비율

In [3]:
print(df["result"].value_counts())
print(df["result"].value_counts(normalize=True) * 100)

result
상품    1861
등외     139
Name: count, dtype: int64
result
상품    93.05
등외     6.95
Name: proportion, dtype: float64


**등외가 6.95%.** 열넷에 하나꼴이다.<br>
공정 데이터의 불량 6.6%와 거의 같다. 소재가 바뀌었을 뿐 **드문 쪽을 맞혀야 하는 구조는 그대로**다.<br>
이 숫자는 Q6에서 바로 쓴다.

## Q3. 정답표를 숫자로 바꾸기

> 선별대 도장은 사람이 읽는 글자다. 그런데 모델은 글자를 못 읽는다.

**문제** — 등외 1, 상품 0인 새 열 만들기 (원래 열은 보존)

In [4]:
df["등외여부"] = (df["result"] == "등외").astype(int)

print(df["등외여부"].value_counts())

등외여부
0    1861
1     139
Name: count, dtype: int64


`df["result"] == "등외"` 가 참/거짓을 만들고, `.astype(int)` 가 참을 1, 거짓을 0으로 바꾼다.<br>
1의 개수가 Q2의 등외 건수와 같으면 제대로 된 것이다.<br>
<br>
**원래 열을 지우지 않은 이유** — 결과를 눈으로 확인할 때 '등외'라는 글자가 있는 편이 읽기 쉽다.<br>
숫자 열은 계산용, 글자 열은 사람용으로 둘 다 둔다.

## Q4. 입력과 정답으로 가르기

> 시험 문제를 만드는 셈이다. 문제지에 답을 적어두면 아무 의미가 없다.

**문제** — 입력은 숫자 센서 8열, 정답은 0/1 열

In [5]:
센서열 = ["temp_avg", "humidity_avg", "co2_ppm", "soil_moisture",
        "ec", "ph", "light_hours", "night_temp_min"]

X = df[센서열]
y = df["등외여부"]

print("입력:", X.shape)
print("정답:", y.shape)

입력: (2000, 8)
정답: (2000,)


**8열인지 반드시 확인한다.** 여기에 `등외여부`나 `result`가 섞여 들어가면<br>
답을 보여주고 답을 맞히라는 셈이 되어 정확도가 100%로 나온다.<br>
잘된 것처럼 보이지만 실제로는 아무것도 학습하지 않은 상태다. 이 사고는 실무에서도 흔하다.<br>
<br>
`batch_id`·`harvested_at`을 뺀 이유도 같다. 상자 번호가 결과를 만들지는 않는다.

## Q5. 학습용과 시험용으로 나누기 ⭐

> "넉 달치를 다 보여주고 맞혔다고 하면, 그건 외운 거 아닌가요?"

**문제** — 80:20 · 다시 실행해도 같게 · 등외 비율 유지

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 시험용 20%
    random_state=42,      # 다시 실행해도 같게
    stratify=y            # 등외 비율을 양쪽에 맞춰서
)

print("학습용:", len(X_train), "| 등외", y_train.sum(), f"({y_train.mean()*100:.2f}%)")
print("시험용:", len(X_test),  "| 등외", y_test.sum(),  f"({y_test.mean()*100:.2f}%)")

학습용: 1600 | 등외 111 (6.94%)
시험용: 400 | 등외 28 (7.00%)


서 반장의 말이 정확히 **과적합** 이야기였다. 넉 달치를 통째로 외우면 새 상자는 못 맞힌다.

| 조건 | 옵션 |
|---|---|
| 20%를 시험용으로 | `test_size=0.2` |
| 다시 실행해도 같게 | `random_state=42` |
| 등외 비율 유지 | `stratify=y` |

**`stratify`가 이 문항의 핵심이다.** 등외가 139건뿐이라 그냥 나누면 한쪽에 몰릴 수 있다.<br>
`random_state`는 42가 아니어도 된다. 아무 숫자나 고정해두면 다시 돌렸을 때 같은 결과가 나온다는 게 요점이다.

## Q6. 아무것도 배우지 않은 기준 모델

> 선별대에 눈 감고 도장만 찍는 사람을 하나 세워본다. 무조건 "상품" 도장.

**문제** — 시험용 전부를 "상품"이라 답했을 때 정확도

In [7]:
정답맞힌수 = (y_test == 0).sum()      # 실제로 상품인 것 = 다 맞힌 것
print("기준 모델 정확도:", 정답맞힌수 / len(y_test) * 100, "%")

기준 모델 정확도: 93.0 %


전부 상품이라 답하면 **실제 상품 372건은 다 맞고, 등외 28건은 다 틀린다.** 372 ÷ 400 = 93%.<br>
<br>
**눈을 감고 있는데 93점이다.** 뒤에서 학습시킨 모델이 93점을 받았다면<br>
그건 잘한 게 아니라 눈 감은 사람과 똑같은 상태라는 뜻이다.<br>
비교할 게 없으면 93점이 좋은 점수인지 알 방법이 없다. 그래서 기준 모델을 먼저 만든다.

## Q7. 진짜 모델 학습시키기

> 눈 감고 찍는 사람 말고, 센서 값을 보고 판단하는 쪽을 세울 차례다.

**문제** — 학습용으로 학습하고 시험용에 예측

In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

모델 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
모델.fit(X_train, y_train)

예측 = 모델.predict(X_test)
print(len(예측), "건 예측 완료")

400 건 예측 완료


**로지스틱 회귀**를 골랐다. 이유는 두 가지.

- 둘 중 하나를 고르는 문제에서 가장 기본이 되는 모델이라 기준으로 삼기 좋다
- "등외일 가능성"을 숫자로 내놓을 수 있다 — Q10에서 문턱을 옮기려면 이게 필요하다

`StandardScaler`를 앞에 붙인 건 **열마다 단위가 달라서**다.<br>
이산화탄소는 600대인데 산도는 6 언저리다. 그대로 넣으면 큰 숫자 쪽이 결과를 좌우한다.<br>
<br>
**의사결정나무나 랜덤포레스트를 골랐어도 맞다.** 그 둘은 크기를 비교하지 않고 자르기만 해서<br>
단위를 맞출 필요가 없다. 다만 숫자는 아래와 조금 다르게 나온다.

## Q8. 혼동행렬 네 칸 채우기 ⭐

> "그래서 몇 개를 놓쳤어요?" — 점수 말고 건수를 묻고 있다.

**문제** — 네 칸이 각각 몇 건인가

In [9]:
from sklearn.metrics import confusion_matrix

행렬 = confusion_matrix(y_test, 예측)
print(행렬)

tn, fp, fn, tp = 행렬.ravel()
print("상품인데 상품이라 함 :", tn)
print("상품인데 등외라 함   :", fp, "  <- 헛경보")
print("등외인데 상품이라 함 :", fn, "  <- 놓친 등외")
print("등외인데 등외라 함   :", tp)

# 검산 — 네 칸의 합이 시험용 건수, 아래 두 칸의 합이 실제 등외 건수
print("\n네 칸 합:", tn + fp + fn + tp, "/ 시험용", len(y_test))
print("아래 두 칸 합:", fn + tp, "/ 실제 등외", int((y_test == 1).sum()))

[[365   7]
 [ 15  13]]
상품인데 상품이라 함 : 365
상품인데 등외라 함   : 7   <- 헛경보
등외인데 상품이라 함 : 15   <- 놓친 등외
등외인데 등외라 함   : 13

네 칸 합: 400 / 시험용 400
아래 두 칸 합: 28 / 실제 등외 28


반장이 물은 답이 여기 있다. **놓친 등외 15건.**<br>
<br>
헛경보 7건은 선별대에서 한 번 더 보면 되지만, 놓친 15건은 **상자째로 나간다.**<br>
<br>
다른 모델을 쓰면 네 칸 숫자는 다르다. 다만 **아래 두 칸의 합이 28**인 것은 어느 모델이든 같아야 한다.<br>
여기가 28이 아니면 나눈 데이터가 아니라 전체로 예측했을 가능성이 크다.

## Q9. 세 가지 지표 구하기 ⭐

> 눈 감고 찍는 사람과 센서를 보는 쪽, 둘을 나란히 세워 본다.

**문제** — 재현율·정밀도·F1을 구하고 기준 모델과 비교

In [10]:
from sklearn.metrics import classification_report

print(classification_report(y_test, 예측,
                            target_names=["상품", "등외"], digits=3))

              precision    recall  f1-score   support

          상품      0.961     0.981     0.971       372
          등외      0.650     0.464     0.542        28

    accuracy                          0.945       400
   macro avg      0.805     0.723     0.756       400
weighted avg      0.939     0.945     0.941       400



In [11]:
# 손으로도 맞춰보기 — 앞 칸에서 받은 네 칸 숫자를 그대로 쓴다
print("재현율:", round(tp / (tp + fn), 3), " <- 실제 등외", tp + fn, "건 중", tp, "건")
print("정밀도:", round(tp / (tp + fp), 3), " <- 등외라 한", tp + fp, "건 중", tp, "건")
print("정확도:", round((tn + tp) / len(y_test), 3))

재현율: 0.464  <- 실제 등외 28 건 중 13 건
정밀도: 0.65  <- 등외라 한 20 건 중 13 건
정확도: 0.945


| | 정확도 | 재현율 | 정밀도 | F1 |
|---|---|---|---|---|
| 기준 모델 (전부 상품) | **93.0%** | **0.000** | 계산 안 됨 | **0.000** |
| 학습시킨 모델 | **94.5%** | **0.464** | 0.650 | **0.542** |

**정확도는 93.0 → 94.5로 겨우 1.5%p 올랐다.** 이것만 보면 "별로 나아진 게 없네" 싶다.<br>
그런데 **재현율은 0에서 0.464로 갔다.** 눈 감고 있던 사람이 이제 등외를 절반 가까이 잡아낸다.<br>
<br>
**기준 모델의 재현율이 0인 이유** — 전부 상품이라 답했으니 등외라고 한 것이 하나도 없다. 0 ÷ 28 = 0.<br>
**정밀도가 "계산 안 됨"인 이유** — 등외라고 한 것이 0건이라 0으로 나누게 된다.<br>
<br>
**모델을 다르게 골랐다면** 대략 이 범위면 정상이다 — 정확도 93~96% · 재현율 0.15~0.6 · 정밀도 0.5~0.9 · F1 0.25~0.65.<br>
등외가 28건뿐이라 한두 건 차이로 지표가 크게 움직인다.

## Q10. 놓친 등외를 더 잡으려면

> "이거 더 잡을 방법은 없어요? 사람이 몇 상자 더 보는 건 괜찮은데."

**문제** — 판정 문턱을 낮춰가며 맞바꿈 보기

In [12]:
# predict_proba - 각 줄이 등외일 "가능성"을 0~1 숫자로 내놓는다
가능성 = 모델.predict_proba(X_test)[:, 1]

for 문턱 in [0.5, 0.3, 0.2, 0.1]:
    판정 = (가능성 >= 문턱).astype(int)      # 문턱을 넘으면 등외로 본다
    tn2, fp2, fn2, tp2 = confusion_matrix(y_test, 판정).ravel()
    print(f"문턱 {문턱} | 잡은 등외 {tp2:2d} | 헛경보 {fp2:2d} | "
          f"재현율 {tp2/(tp2+fn2):.3f} | 정밀도 {tp2/(tp2+fp2):.3f}")

문턱 0.5 | 잡은 등외 13 | 헛경보  7 | 재현율 0.464 | 정밀도 0.650
문턱 0.3 | 잡은 등외 19 | 헛경보 13 | 재현율 0.679 | 정밀도 0.594
문턱 0.2 | 잡은 등외 21 | 헛경보 19 | 재현율 0.750 | 정밀도 0.525
문턱 0.1 | 잡은 등외 25 | 헛경보 36 | 재현율 0.893 | 정밀도 0.410


**모델은 그대로다.** 다시 학습시키지 않았다. 바꾼 건 "얼마나 의심스러우면 등외로 볼까" 하나뿐이다.

- 잡은 등외 13 → 25건 (재현율 0.464 → 0.893)
- 헛경보 7 → 36건 (정밀도 0.650 → 0.410)

**한쪽이 좋아지면 한쪽이 나빠진다.** 둘 다 올리는 문턱은 없다.<br>
<br>
문턱 0.1에서는 정확도가 오히려 90% 근처로 떨어져 기준 모델(93%)보다 낮아진다.<br>
그런데도 이 판정이 더 나을 수 있다. **무엇을 볼지 먼저 정해두라**고 한 이유가 이것이다.

---
## 마지막 — 반장의 마지막 질문

> "그러니까 **어디까지 잡을지는 제가 정하는 거네요.**"

맞다. 데이터는 **선택지와 그 대가**를 보여줄 뿐이고, 어느 쪽이 더 아픈지는 현장이 정한다.<br>
Q10의 표가 딱 그것이다. 네 줄 중 어느 것도 정답이 아니고, **네 줄 전부가 선택지**다.

## 소재만 다르지 푸는 방식은 같다

| | 오늘 수업 (반도체 공정) | 이 미션 (스마트팜 온실) |
|---|---|---|
| 한 줄이 뜻하는 것 | 한 번의 생산 흐름 | 한 상자의 출하 |
| 조건 열 | 센서 590개 → 추린 것 | 센서 8개 |
| 결과 열 | 양품 / 불량 | 상품 / 등외 |
| 드문 쪽 비율 | 6.6% | 6.95% |
| 기준 모델 정확도 | 약 93% | 93.0% |
| 하는 일 | 나누고 → 기준 모델 → 학습 → 혼동행렬 → 지표 | **똑같음** |

3주차에 각자 데이터를 고를 때 이 구조를 찾으면 된다 — **조건 열 몇 개 + 결과 열 하나.**